<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Greedy Tour Construction

**Construct a feasible tour by repeatedly choosing the nearest unvisited city.**

Keep the five-city symmetric distance matrix, the requirement to visit every city once and return, and the total-distance objective. The decision is a discrete visit order. The clock observations, calibration equality, and correction bound do not apply to this delivery system. The new procedure commits to one immediate choice at a time; changing its construction start can change the final tour.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Build a tour with a greedy rule

A **heuristic** searches for a useful solution without a general guarantee of finding the global optimum. It still has to respect the problem's constraints. A **greedy construction** commits to the best immediate choice under a stated rule, without revising earlier choices.

The **nearest-neighbor rule** chooses the closest unvisited city. If the current city is $i$ and $U$ is the set of unvisited cities, choose

$$
j^\star\in\underset{j\in U}{\operatorname{argmin}}\ C_{ij}.
$$

Here $\operatorname{argmin}$ returns a city attaining the smallest distance. We break ties by choosing the smaller city label. After every city is visited, return to the starting city.

Compare the four labeled distances from city 2. The orange edge selects city 4 because its distance is smallest. This picture shows one choice; line lengths do not represent the distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/greedy_next_city.png" alt="One greedy step from city 2 compares distances 9, 8, 4, and 3 to unvisited cities 0, 1, 3, and 4. The orange arrow selects city 4 at distance 3." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Starting at city 2, as in the supplied code, the rule gives:

| Current city | Unvisited cities | Chosen next city | Added distance |
|:---|:---|:---|---:|
| 2 | 0, 1, 3, 4 | 4 | 3 |
| 4 | 0, 1, 3 | 3 | 1 |
| 3 | 0, 1 | 1 | 5 |
| 1 | 0 | 0 | 2 |
| 0 | None | Return to 2 | 9 |

The cheap first edge does not account for the later return edge. The rule constructs a feasible tour because every pair is connected, each new city is unvisited, and the route closes at the end. These facts do not prove that its total distance is smallest.


In [ ]:
def nearest_neighbor(start_city):
    if start_city not in range(CITY_COUNT):
        raise ValueError("The starting city must be one of the city labels.")
    remaining = set(range(CITY_COUNT)) - {start_city}
    tour = [start_city]
    trace = []
    while remaining:
        current = tour[-1]
        next_city = min(remaining,
                        key=lambda city: (DISTANCES[current, city], city))
        trace.append({"current": current, "unvisited": tuple(sorted(remaining)),
                      "chosen": next_city,
                      "distance": int(DISTANCES[current, next_city])})
        tour.append(next_city)
        remaining.remove(next_city)
    tour.append(start_city)
    result = evaluate_tour(tour)
    result["trace"] = trace
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Run the original starting choice. The result should reproduce the worked total of 20 distance units, including the return edge.


In [ ]:
source_run = nearest_neighbor(2)
for step in source_run["trace"]:
    print(step)
print("Tour:", source_run["decision"])
print("Return distance:", source_run["edge_distances"][-1])
print("Total distance:", source_run["objective"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Change only the construction start

Keep the distance matrix, visit requirements, and objective fixed. Change only the city from which the algorithm begins constructing a tour.

In this problem the physical tour is a cycle: rotating its written starting point does not change its edges or distance. No depot, time window, or required departure city is imposed. The construction start is therefore a **hyperparameter** of this search. It can change the selected tour; it cannot change the distance of a fixed tour.

For example, starting at city 1 gives $(1,0,4,3,2,1)$, with distance $2+7+1+4+8=22$. The tour from city 2 has distance 20, so the first result is already a counterexample to a general optimality claim for this greedy rule.


In [ ]:
greedy_results = [nearest_neighbor(start) for start in range(CITY_COUNT)]
for start, result in enumerate(greedy_results):
    print(f"Start {start}: {result['decision']} | distance {result['objective']}")
best_greedy = min(greedy_results, key=lambda result: result["objective"])
print("Best among these five greedy runs:", best_greedy["objective"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook develops the nearest-neighbor choices and compares all five construction starts.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked calculations use the supplied matrix.
